# Extraction and Transformation Workflow

This notebook demonstrates the complete workflow for extracting and transforming documents using Dacke:
1. Start the app in the background
2. Define the workspace name
3. Get the default/production pipeline
4. Run extraction and transformation with a given URI

In [1]:
import logging

import sys

sys.path.append("src")

logging.basicConfig(level=logging.INFO)


In [2]:
import json

import httpx

In [3]:
base_url: str = "http://127.0.0.1:8000/api/v1"
workspace_name: str = "example_workspace"
collection_name: str = "example_collection"
pipeline_id: str = '751608e4ffdd57a0a29195b14bf9cb4c'

In [ ]:
async with httpx.AsyncClient(timeout=30) as client:

    # --- List existing workspaces ---
    resp = await client.get(f"{base_url}/workspaces")
    resp.raise_for_status()
    workspaces = resp.json()

    workspace_id = next((w["id"] for w in workspaces if w["name"] == workspace_name), None)
    if workspace_id:
        print(f"Workspace '{workspace_name}' already exists (ID: {workspace_id}).")
        raise Exception("Workspace already exists")


    workspace = await client.post(f"{base_url}/workspaces", json={"name": workspace_name})

    if not workspace.is_success:
        print("Failed to create workspace:", workspace.text)
        raise Exception("Failed to create workspace")

    workspace = workspace.json()
    collection = await client.post(
        f"{base_url}/workspaces/{workspace.get('id')}/collections",
        json={
            "workspace_id": workspace.get("id"),
            "name": collection_name
        },
    )

    collection = collection.json()
    if not collection:
        print("Failed to create collection:", collection.text)
        raise Exception("Failed to create collection")


    pipeline_resp = await client.get(
        f"{base_url}/workspaces/{workspace.get('id')}/collections/{collection.get('id')}/pipelines/production"
    )
    if not pipeline_resp.is_success:
        print("Failed to retrieve pipeline:", pipeline_resp.text)
        raise Exception("Failed to retrieve pipeline")
    
    pipeline = pipeline_resp.json()
    id = pipeline.get("id")

print(f"Pipeline for collection '{collection_name}': {json.dumps(pipeline, indent=2)}")

In [4]:
from dacke.domain.aggregates.pipeline import Pipeline
from dacke.domain.values.collection import CollectionID
from dacke.domain.values.extraction import (
    ExtractionSettings,
    TableSettings,
    ImageSettings,
    EnrichmentSettings,
)

pipeline = Pipeline.create(
    name="Example Pipeline",
    collection_id=CollectionID.generate(),
    extraction_settings=ExtractionSettings(
        tables=TableSettings(
            enabled=True,
            quality="accurate",
        ),
        images=ImageSettings(
            scale=2.0,
            page_images=True,
            picture_images=True,
            parsed_pages=False,
        ),
        enrichments=EnrichmentSettings(
            code=False,
            formulas=False,
            picture_classification=True,
            picture_description=False,
        ),
    ),
)

In [5]:
from logging import getLogger

from docling.datamodel.accelerator_options import AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    CodeFormulaVlmOptions,
    EasyOcrOptions,
    PdfPipelineOptions,
    PictureDescriptionApiOptions,
    PictureDescriptionVlmEngineOptions,
    TableFormerMode,
    TableStructureOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
    TripletTableSerializer,
)
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.serializer.markdown import MarkdownParams
from docling_core.types.doc.base import ImageRefMode
from docling_core.types.doc.document import DoclingDocument
from pydantic import AnyUrl, HttpUrl
from transformers import AutoTokenizer

from dacke.application.ports.extractor import Extractor
from dacke.domain.aggregates.document import Document
from dacke.domain.entities.chunk import Chunk
from dacke.domain.exceptions import DomainError
from dacke.domain.values.chunk import ChunkID
from dacke.domain.values.document import DocumentID, DocumentMetadata
from dacke.domain.values.extraction import ExtractionSettings
from docling_core.types.doc.document import DoclingDocument, PictureItem, TableItem, CodeItem
from dacke.domain.values.pipeline import PipelineID
from dacke.domain.entities.attachment import Attachment
from dacke.domain.values.attachment import AttachmentTypes
from dacke.domain.values.artifact import StoragePath, ObjectAddress
from dacke.domain.values.artifact import ArtifactID
from dacke.domain.values.attachment import Content



logging = getLogger(__name__)


class Serializer(ChunkingSerializerProvider):
    """Custom serializer for document chunking with image handling."""

    def get_serializer(self, doc: DoclingDocument) -> ChunkingDocSerializer:
        return ChunkingDocSerializer(
            doc=doc,
            table_serializer=TripletTableSerializer(),
            params=MarkdownParams(
                image_placeholder="![image]",
                image_mode=ImageRefMode.REFERENCED,
                indent=4,
            ),
        )


class Extractor(Extractor[ExtractionSettings, Document]):

    def pdf(self, request: ExtractionSettings) -> PdfPipelineOptions:
        options = PdfPipelineOptions()
        options.document_timeout = request.runtime.document_timeout
        options.artifacts_path = request.runtime.artifacts_path
        options.allow_external_plugins = request.runtime.allow_external_plugins
        options.force_backend_text = request.runtime.force_backend_text

        options.accelerator_options = AcceleratorOptions(
            device=request.compute.device,
            num_threads=request.compute.num_threads,
            cuda_use_flash_attention2=False,
        )

        options.do_ocr = request.ocr.enabled
        if request.ocr.enabled:
            options.ocr_options = EasyOcrOptions(
                lang=request.ocr.languages,
                force_full_page_ocr=False,
                bitmap_area_threshold=0.05,
                use_gpu=None,
            )

        options.do_table_structure = request.tables.enabled
        options.table_structure_options = TableStructureOptions(
            do_cell_matching=True,
            mode=(
                TableFormerMode.ACCURATE
                if request.tables.quality in {"accurate", "balanced"}
                else TableFormerMode.FAST
            ),
        )

        options.do_code_enrichment = request.enrichments.code
        options.do_formula_enrichment = request.enrichments.formulas
        options.code_formula_options = CodeFormulaVlmOptions.from_preset(
            "codeformulav2",
            scale=2.0,
            max_size=None,
            extract_code=request.enrichments.code,
            extract_formulas=request.enrichments.formulas,
        )

        options.images_scale = request.images.scale
        options.generate_page_images = request.images.page_images
        options.generate_picture_images = request.images.picture_images
        options.generate_parsed_pages = request.images.parsed_pages

        options.do_picture_classification = request.enrichments.picture_classification

        options.do_picture_description = request.enrichments.picture_description
        if request.enrichments.picture_description:
            if request.description.use_remote_api:
                options.enable_remote_services = True
                options.picture_description_options = PictureDescriptionApiOptions(
                    url=AnyUrl(request.description.url),
                    headers=request.description.headers,
                    params=request.description.params,
                    timeout=request.description.timeout,
                    concurrency=request.description.concurrency,
                    prompt=request.description.prompt,
                    scale=2.0,
                    picture_area_threshold=0.5,
                    classification_min_confidence=0.0,
                )
            else:
                options.picture_description_options = (
                    PictureDescriptionVlmEngineOptions.from_preset(
                        "smolvlm",
                        prompt=request.description.prompt,
                        scale=2.0,
                        picture_area_threshold=0.5,
                        classification_min_confidence=0.0,
                    )
                )

        options.ocr_batch_size = request.runtime.ocr_batch_size
        options.layout_batch_size = request.runtime.layout_batch_size
        options.table_batch_size = request.runtime.table_batch_size
        options.batch_polling_interval_seconds = (
            request.runtime.batch_polling_interval_seconds
        )
        options.queue_max_size = request.runtime.queue_max_size

        return options



    def chunker(
        self, 
        config: ExtractionSettings
    ) -> HybridChunker:
        
        serializer = Serializer()

        tokenizer = HuggingFaceTokenizer(
            tokenizer=AutoTokenizer.from_pretrained(
                "sentence-transformers/all-MiniLM-L6-v2",
                use_fast=True,
            ),
            max_tokens=512
        )

        chunker = HybridChunker(
            tokenizer=tokenizer,
            serializer_provider=serializer,
            merge_peers=True,
        )
        return chunker

    def _extract_image_attachments(
            self, 
            document: DoclingDocument, 
            folder: StoragePath, 
            pipeline_id: PipelineID, 
        ) -> dict[str, Attachment]:
        """Extract images from the document and replace them with references."""
        attachments = {}
        for idx, item in enumerate(document.tables):
            content = item.export_to_dataframe(document)
            assert content is not None, f"Failed to extract image content for TableItem {item.self_ref}"

            attachment = Attachment.create(
                pipeline_id = pipeline_id,
                location = folder.resolve(f"table_{idx}.csv"),
                attachment_type = AttachmentTypes.TABLE,
                metadata={
                    "caption": item.caption_text(document)
                },
                content = Content.from_csv(content)
            )
            logging.info(item.model_dump_json(indent=2))

            assert attachment is not None, f"Failed to create attachment for TableItem {item.self_ref}"
            attachments[item.self_ref] = attachment


        return attachments
    
    def _extract_table_attachments(
            self, 
            document: DoclingDocument, 
            folder: StoragePath, 
            pipeline_id: PipelineID, 
        ) -> dict[str, Attachment]:
        """Extract images from the document and replace them with references."""
        attachments = {}
        for idx, item in enumerate(document.pictures):
            location = folder.resolve(f"picture_{idx}.jpg")
            content = item.get_image(document)

            assert content is not None, f"Failed to extract image content for TableItem {item.self_ref}"
            assert item.image is not None and item.image.uri is not None, f"PictureItem {item.self_ref} is missing image URI"
            item.image.uri = AnyUrl(location.s3_uri)

            attachment = Attachment.create(
                pipeline_id = pipeline_id,
                location = location,
                attachment_type = AttachmentTypes.IMAGE,
                metadata={
                    "caption": item.caption_text(document),
                    "description": getattr(item.meta, "description", None),
                },
                content = Content.from_image(content)
            )
            assert attachment is not None, f"Failed to create attachment for PictureItem {item.self_ref}"
            attachments[item.self_ref] = attachment

        return attachments
    
    def _extract_attachments(
            self, 
            document: DoclingDocument, 
            folder: StoragePath, 
            pipeline_id: PipelineID,
    ):
        image_attachment =self._extract_image_attachments(
            document=document,
            folder=folder.at("images"),
            pipeline_id=pipeline_id,
        )

        table_attachment = self._extract_table_attachments(
            document=document,
            folder=folder.at("tables"),
            pipeline_id=pipeline_id,
        )
        return {**image_attachment, **table_attachment}

    async def extract(
            self, 
            pipeline_id: PipelineID,
            folder: StoragePath,
            extraction_settings: ExtractionSettings, 
            url: AnyUrl
        ) -> Document:


        result = Document(
            identity=DocumentID.from_ref(
                "filename", 
                pipeline_id=pipeline_id
            ),
            metadata=DocumentMetadata(
                title="#Filename",
                source_url=url.encoded_string()
            ),
        )

        logging.info(f"Starting extraction for URL: {url} with config: {extraction_settings}")
        converter = DocumentConverter(
            allowed_formats=[
                InputFormat.PDF,
                InputFormat.DOCX,
                InputFormat.PPTX,
                InputFormat.HTML,
                InputFormat.IMAGE,
                InputFormat.ASCIIDOC,
                InputFormat.MD,
                InputFormat.CSV,
                InputFormat.JSON_DOCLING,
                InputFormat.AUDIO,
                InputFormat.VTT,
                InputFormat.LATEX,
            ],
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=self.pdf(extraction_settings))
            },
        )

        conversion = converter.convert(str(url))
        doc = conversion.document

        # Ensure document was successfully converted
        if doc is None:
            raise DomainError(f"Failed to convert document {url}: No document produced")

        chunker = self.chunker(extraction_settings)
        attachments = self._extract_attachments(
            document=doc,
            folder=folder,
            pipeline_id=pipeline_id,
        )

        assert doc is not None, "Document conversion failed, no document produced"
        for idx, element in enumerate(chunker.chunk(doc)):
        
            chunk = Chunk.create(
                content=element.text,
                document_id=result.identity,
                reference=f"#Filename/#chunk-{idx}"
            )

            assert element.meta is not None, f"Chunk {idx} is missing metadata"

            for item in getattr(element.meta, "doc_items", []):
                reference = getattr(item, "self_ref", None)
                assert reference is not None, f"Doc item in chunk {idx} is missing self_ref"
                if reference in attachments:
                    chunk.add_attachment(attachments.pop(reference))

            result.add_chunk(chunk)


        return result
    

/Users/jonasmeddeb/Documents/rag/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from dacke.domain.values.artifact import StoragePath

folder = StoragePath(
    bucket="collection",
    prefix="/pipelines/files"
)

extractor = Extractor()
doc = await extractor.extract(
    pipeline_id=pipeline.identity,
    folder=folder,
    extraction_settings=pipeline.extraction_settings,   
    url=HttpUrl("https://arxiv.org/pdf/2408.09869"))

INFO:__main__:Starting extraction for URL: https://arxiv.org/pdf/2408.09869 with config: compute=ComputeSettings(device='auto', num_threads=4) ocr=OcrSettings(enabled=True, languages=['en']) tables=TableSettings(enabled=True, quality='accurate') enrichments=EnrichmentSettings(code=False, formulas=False, picture_classification=True, picture_description=False) images=ImageSettings(scale=2.0, page_images=True, picture_images=True, parsed_pages=False) description=DescriptionServiceSettings(use_remote_api=False, url='http://localhost:1234/v1/chat/completions', headers={}, params={'model': 'qwen/qwen3-vl-8b', 'seed': 42, 'max_completion_tokens': 200}, timeout=90.0, concurrency=1, prompt='Describe this image in a few sentences. Be concise and accurate.') runtime=PipelineRuntimeSettings(document_timeout=120.0, artifacts_path=None, allow_external_plugins=False, force_backend_text=False, ocr_batch_size=4, layout_batch_size=4, table_batch_size=4, batch_polling_interval_seconds=0.5, queue_max_size

In [7]:
print(doc.model_dump_json(indent=2))

{
  "identity": {
    "value": "e6ee4176-8718-528b-9e55-a6a3df1f4a79"
  },
  "metadata": {
    "title": "#Filename",
    "source_url": "https://arxiv.org/pdf/2408.09869"
  },
  "chunks": [
    {
      "identity": {
        "value": "96c42fbc-4bae-54e2-9127-889c5d055877"
      },
      "document_id": {
        "value": "e6ee4176-8718-528b-9e55-a6a3df1f4a79"
      },
      "content": "Icon\n\n![Image](s3://collection//pipelines/files/tables/picture_0.jpg)",
      "metadata": {
        "order": null,
        "pages": null,
        "title": null
      },
      "attachments": [
        {
          "identity": {
            "value": "e7ed4e33-e9fc-5c3b-849f-e0cf7e545517"
          },
          "location": {
            "path": {
              "bucket": "collection",
              "prefix": "/pipelines/files/tables"
            },
            "filename": "picture_0.jpg",
            "version_id": null
          },
          "type": "image",
          "metadata": {
            "caption": "",
 

In [14]:
# plot png
data = doc.chunks[0].attachments[0].data
if data is not None:
    print(data.data[:100])  # Print the first 100 bytes of the image data
    from PIL import Image
    from io import BytesIO

    image = Image.open(BytesIO(data.data))
    image.show()

b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\xc6\x00\x00\x00\xc0\x08\x02\x00\x00\x00\xd0\xa0\x8b\x17\x00\x00\xa0WIDATx\x9c\xec\xbd\x07\x9c\\U\xd9?~\xca-Sw\xb6\xd7d\xb3\xbb\xd9\xf4Bz\x08\xa1w\x90.\x8aHQyi"\x06\x94\xd7\x8a"\n\x8a\x80bC\x14\x01\x11A\x90\xd0{\xef\xa4\xf7\x90\x90\xb6\xd9'


In [16]:
doc.chunks[0].attachments[0].location.s3_uri

's3://collection//pipelines/files/tables/picture_0.jpg'